# Experiments 

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import mlflow
import shap



from sklearn.preprocessing import PowerTransformer, OneHotEncoder, MultiLabelBinarizer, OrdinalEncoder, MinMaxScaler, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import KNNImputer, SimpleImputer, IterativeImputer, MissingIndicator
from category_encoders import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_validate
from sklearn.neighbors import LocalOutlierFactor

In [2]:
import dagshub
dagshub.init(repo_owner='bowlekarbhushan88', repo_name='property-price-prediction', mlflow=True)

Accessing as bowlekarbhushan88

Initialized MLflow to track repo "bowlekarbhushan88/property-price-prediction"

Repository bowlekarbhushan88/property-price-prediction initialized!

In [3]:
# set the tracking server

mlflow.set_tracking_uri("https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow")

In [4]:
#Here, I used the data saved using VS Code. This data has 184 columns, which means it is the result of merging df and am_df.
df = pd.read_csv(r'C:\Users\AMD\Desktop\bhushan PC data 11-8-2025/bhushan/property_project/files_vscode/data/py_cleaned_data.csv')

C:\Users\AMD\AppData\Local\Temp\ipykernel_30628\1989270230.py:2: DtypeWarning: Columns (106,128,134,135,136,137,138,139) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r'C:\Users\AMD\Desktop\bhushan PC data 11-8-2025/bhushan/property_project/files_vscode/data/py_cleaned_data.csv')


In [5]:
# drop columns not required for model input 
columns_to_drop = ['id','price_category','costpersqft','emi']

df.drop(columns = columns_to_drop , inplace = True)

In [6]:
# combine all amenities and make one column
# Step 1: Filter columns that start with 'am_'
am_cols = [col for col in df.columns if col.startswith('am_')]

# Step 2: Combine values row-wise into a comma-separated string (not a list)
df['amenities'] = df[am_cols].apply(
    lambda row: ', '.join([str(val).strip() for val in row if isinstance(val, str) and val.strip() != ""]),
    axis=1
)

In [7]:
# Step 3: Drop original 'am_' columns
df.drop(columns=am_cols, inplace=True)

In [8]:
#drop duplicate rows 
df.drop_duplicates(inplace=True)

## Data preparation 

In [9]:
temp_df = df.copy()

In [10]:
# split data 

X = temp_df.drop(columns = 'price')
y = temp_df['price']

In [11]:
# train test split

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
print("The size of train data is",X_train.shape)
print("The shape of test data is",X_test.shape)

The size of train data is (9302, 47)
The shape of test data is (2326, 47)


`lets categorized the columns`

## Transform Target column

In [13]:
pt = PowerTransformer(method='yeo-johnson')
y_train_trans = pd.Series(
    pt.fit_transform(y_train.values.reshape(-1, 1)).ravel(),
    index=y_train.index
)
y_test_trans = pd.Series(
    pt.transform(y_test.values.reshape(-1, 1)).ravel(),
    index=y_test.index
)

In [14]:
#o/p in daraframe
from sklearn import set_config
set_config(transform_output="pandas")

`Till this the code is common for all below experimants`

# experiment :01 (Baseline_model)

## Imputation Pipeline 

so i have asked one question(on 23-7-25) to sir for clipping and once you get answer then make changes according to that and then ask chatgpt 
as which imputation is better by uploading graph and this also

this only a sample you have to put this below for orginal , KNN and iterative also    
 count    6512.000000  
mean        3.152906  
std         1.595011  
min         1.304996  
25%         2.621082  
50%         2.752291  
75%         3.100318  
max        15.646208  
Name: project_name, dtype: float64 


In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 1 - Baseline model Using Random Forest")

<Experiment: artifact_location='mlflow-artifacts:/f921f82952fb42099abb865b745a059d', creation_time=1754360469802, experiment_id='0', last_update_time=1755069559430, lifecycle_stage='active', name='Exp 1 - Baseline model Using Random Forest', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
#done
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X

In [20]:
#done
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode   
'project_name' - constant / top 200 / target encode  
'location' - constant / top 200 / target encode  


'project_in_acres' - KNN Imputer    
'area' - KNN Imputer   
'education_mean_km' - KNN Imputer  
'education_min_km' - KNN Imputer  
'transport_mean_km' - KNN Imputer  
'transport_min_km' - KNN Imputer  
'shopping_centre_mean_km' - KNN Imputer  
'shopping_centre_min_km' - KNN Imputer  
'overall_min_mean_km' - KNN Imputer  
'overall_avg_mean_km' - KNN Imputer  
'overall_min_min_km' - KNN Imputer  
'overall_avg_min_km' - KNN Imputer  
'available_units' - KNN Imputer  
'towers' - KNN Imputer  
'flat_on_floor' - KNN Imputer  
'total_floor' - KNN Imputer  
'bath' - KNN Imputer  
'parking' - KNN Imputer  
'commercial_hub_mean_km' - KNN Imputer  
'commercial_hub_min_km' - KNN Imputer  
'balcony' - KNN Imputer  

'lattitude' - iterative imputer   
'longitude' - iterative imputer   

'lift' - median  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  
'furnish' - mode / ordinal encoding  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ('top_k', TopKCategoriesTransformer(top_k=200)),
    ('target_encoder', TargetEncoder())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 ))
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories))
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation
    (KNNImputer(n_neighbors=5), features_to_fill_knn),
    (IterativeImputer(), features_to_fill_iterative),
    (SimpleImputer(strategy="median"), features_to_fill_median),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

`observation`

After transforming the target variable using PowerTransformer, the model predicted some values that were slightly outside the range learned during training. When applying the inverse transformation, these out-of-range values caused NaN outputs.

To fix this, I used clipping to limit the predicted values within the min and max range of the transformed training data. This ensured the inverse transformation worked properly without returning any NaN values.

Only 2 values were affected, so clipping was a safe and effective solution.

`observation`

- from MAE on training data get 18 lakhs error and on test data get 42 lakhs error
- from RMSE on training data get 64 lakhs error and on test data get 1.03 cr error
- r2 score near to 1 which is good

In [25]:
X_train_trans = final_pipeline.fit_transform(X_train,y_train_trans)
X_test_trans = final_pipeline.transform(X_test)

#print(X_train_trans.head())
#print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])


In [26]:
# Model setup
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)


# Fit model
rf.fit(X_train_trans, y_train_trans)


# Predict & inverse transform
y_pred_train_trans = rf.predict(X_train_trans)
y_pred_test_trans = rf.predict(X_test_trans)


# Clip before inverse transform
min_val, max_val = y_train_trans.min(), y_train_trans.max()
y_pred_train = pt.inverse_transform(np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)).ravel()
y_pred_test = pt.inverse_transform(np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)).ravel()


# Train/Test evaluation
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

train_metrics = calc_metrics(y_train, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)


# Cross-validation evaluation
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    rf,
    X_train_trans,
    y_train_trans,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)


# Print all results neatly
print("==== Train Metrics ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")
print("\n==== Test Metrics ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")


==== Train Metrics ====
MAE: 0.4925 | MSE: 3.4869 | RMSE: 1.8673 | R²: 0.8107

==== Test Metrics ====
MAE: 0.5604 | MSE: 3.3080 | RMSE: 1.8188 | R²: 0.7739

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1439, Std: 0.0007
train_MSE - Mean: 0.0395, Std: 0.0004
train_RMSE - Mean: 0.1988, Std: 0.0011
train_R2 - Mean: 0.9605, Std: 0.0007

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1959, Std: 0.0038
test_MSE - Mean: 0.0729, Std: 0.0031
test_RMSE - Mean: 0.2700, Std: 0.0058
test_R2 - Mean: 0.9270, Std: 0.0048


In [27]:
# Now feature importances can be extracted using X_train_trans.columns
importances = rf.feature_importances_
feature_names = X_train_trans.columns


# Create DataFrame of importances
feat_imp_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

# Display all features
pd.set_option("display.max_rows", None)
print(feat_imp_df)

                              Feature    Importance
82                                bed  1.711606e-01
19                               bath  1.441018e-01
4                                area  1.440401e-01
2                            location  9.049225e-02
68                        city_mumbai  7.848970e-02
24                          lattitude  3.964286e-02
20                            parking  3.849281e-02
25                          longitude  3.797014e-02
1                        project_name  2.446042e-02
18                        total_floor  2.383443e-02
0                             builder  2.251638e-02
70                       city_palghar  1.784473e-02
52                extra_rooms_servant  1.208634e-02
22              commercial_hub_min_km  9.804604e-03
21             commercial_hub_mean_km  9.285919e-03
17                      flat_on_floor  8.268443e-03
71                         city_thane  8.083999e-03
81                   total_within_2km  8.025540e-03
73          

In [28]:
# # 1. Initialize the SHAP explainer
# explainer = shap.TreeExplainer(rf)  # rf is your trained RandomForestRegressor

# # 2. Calculate SHAP values for the training data
# shap_values = explainer.shap_values(X_train_trans)

# # 3. Summary plot (bar chart of feature importance)
# shap.summary_plot(shap_values, X_train_trans, plot_type="bar")

# # 4. Full summary plot (beeswarm plot)
# shap.summary_plot(shap_values, X_train_trans)

`observation`
- In SHAP summary plots, positive SHAP values push the model prediction (property price) higher and negative values push it lower. Color shows the feature value — red for high and blue for low.
- wider spread in SHAP values, contribute more to predicting the target. As move down the plot, the spread of SHAP values decreases, which means their contribution to the target prediction also reduces accordingly. so hence we can drop the features after total_within_2km column and train the model again  
- Positive SHAP values → Increase the predicted property price  
and Negative SHAP values → Decrease the predicted property price    
- High values (red) push SHAP values to the right → i.e., increase price. So, larger area,bed higher predicted price (as expected).
- When the city is Mumbai (value = 1, shown in red), it increases price. When it's not Mumbai (value = 0, shown in blue), it lowers price.  
- Higher number of total floors (red) slightly push the price higher (positive SHAP). Lower values (blue) push price lower.
- Properties located at lower latitude and longitude usually have higher prices.
- Properties in Palghar usually have lower prices.

In [29]:
# #force plot

# # Initialize the JS visualization
# shap.initjs()

# # Create force plot for all rows
# shap.force_plot(
#     explainer.expected_value,  # scalar or array of expected values
#     shap_values,               # SHAP values for all samples
#     X_train_trans              # Feature values
# )


In [30]:
# def plot_shap_waterfall(explainer, shap_values, X_data, row_index):

#     # Extract row data and corresponding SHAP values
#     row_data = X_data.iloc[row_index]
#     row_shap_values = shap_values[row_index]

#     # Generate waterfall plot
#     shap.plots._waterfall.waterfall_legacy(
#         base_value=explainer.expected_value,
#         shap_values=row_shap_values,
#         features=row_data,
#         feature_names=row_data.index
#     )



In [31]:
# # Example usage for row 5
# plot_shap_waterfall(explainer, shap_values, X_train_trans, row_index=5)

In [32]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Log experiment
with mlflow.start_run(run_name="Baseline model Using Random Forest"):
    # Log experiment type
    mlflow.log_param("experiment_type", "Baseline model Using Random Forest")

    # Log model parameters
    mlflow.log_params(rf.get_params())

    # Log train/test evaluation metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log mean cross-validation metrics
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))

🏃 View run Baseline model Using Random Forest at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/0/runs/8e9f2035b2dc4b7eb23d0a1b2f4b0b2e
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/0


# experiment :02 (Baseline_model + Added scaling techniques for the selected columns.)

In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 2 - Baseline model and scaling for selected columns")

<Experiment: artifact_location='mlflow-artifacts:/cf244dc7c7d043408e97c8179021fa89', creation_time=1754570556325, experiment_id='2', last_update_time=1755070808116, lifecycle_stage='active', name='Exp 2 - Baseline model and scaling for selected columns', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
#done
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X

In [20]:
#done
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode / standardization  
'project_name' - constant / top 200 / target encode  / standardization  
'location' - constant / top 200 / target encode  / standardization  


'project_in_acres' - KNN Imputer / standardization      
'area' - KNN Imputer  / standardization   
'education_mean_km' - KNN Imputer  / standardization  
'education_min_km' - KNN Imputer  / standardization  
'transport_mean_km' - KNN Imputer  / standardization  
'transport_min_km' - KNN Imputer  / standardization  
'shopping_centre_mean_km' - KNN Imputer  / standardization  
'shopping_centre_min_km' - KNN Imputer  / standardization  
'overall_min_mean_km' - KNN Imputer  / standardization  
'overall_avg_mean_km' - KNN Imputer  / standardization  
'overall_min_min_km' - KNN Imputer  / standardization  
'overall_avg_min_km' - KNN Imputer  / standardization  
'available_units' - KNN Imputer  / standardization  
'towers' - KNN Imputer   / standardization   
'total_floor' - KNN Imputer  / standardization  
'bath' - KNN Imputer  / standardization   
'parking' - KNN Imputer  / standardization  
'commercial_hub_mean_km' - KNN Imputer  / standardization  
'commercial_hub_min_km' - KNN Imputer  / standardization  
'balcony' - KNN Imputer / standardization  

'flat_on_floor' - KNN Imputer 


'lattitude' - iterative imputer   / standardization  
'longitude' - iterative imputer   / standardization  

'lift' - median  / standardization  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  / standardization  
'furnish' - mode / ordinal encoding  / standardization  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode 

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn_standardization = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_knn = ['flat_on_floor']

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("top_k", TopKCategoriesTransformer(top_k=200)),
    ("target_encoder", TargetEncoder()),
    ("scaler", StandardScaler())  
])

knn_then_standardize = Pipeline(steps=[
    ("knn_imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

iterative_then_standardize = Pipeline(steps=[
    ("iterative_imputer", IterativeImputer()),
    ("scaler", StandardScaler())
])

median_then_standardize = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 )),
    ('scaler', StandardScaler())
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories)),
    ('scaler', StandardScaler())
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation and standardization
    (knn_then_standardize, features_to_fill_knn_standardization),
    (iterative_then_standardize, features_to_fill_iterative),
    (median_then_standardize, features_to_fill_median),
    
    #imputattion
    (KNNImputer(n_neighbors=5), features_to_fill_knn),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

In [25]:
X_train_trans = final_pipeline.fit_transform(X_train,y_train_trans)
X_test_trans = final_pipeline.transform(X_test)

#print(X_train_trans.head())
#print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])


In [26]:
# Model setup
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)


# Fit model
rf.fit(X_train_trans, y_train_trans)


# Predict & inverse transform
y_pred_train_trans = rf.predict(X_train_trans)
y_pred_test_trans = rf.predict(X_test_trans)

# Clip before inverse transform
min_val, max_val = y_train_trans.min(), y_train_trans.max()
y_pred_train = pt.inverse_transform(np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)).ravel()
y_pred_test = pt.inverse_transform(np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)).ravel()


# Train/Test evaluation
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

train_metrics = calc_metrics(y_train, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)


# Cross-validation evaluation
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    rf,
    X_train_trans,
    y_train_trans,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)


# Print all results neatly
print("==== Train Metrics ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")
print("\n==== Test Metrics ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")


==== Train Metrics ====
MAE: 0.4870 | MSE: 3.4409 | RMSE: 1.8550 | R²: 0.8132

==== Test Metrics ====
MAE: 0.5585 | MSE: 3.2245 | RMSE: 1.7957 | R²: 0.7796

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1443, Std: 0.0008
train_MSE - Mean: 0.0397, Std: 0.0005
train_RMSE - Mean: 0.1992, Std: 0.0012
train_R2 - Mean: 0.9603, Std: 0.0007

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1963, Std: 0.0030
test_MSE - Mean: 0.0733, Std: 0.0023
test_RMSE - Mean: 0.2706, Std: 0.0043
test_R2 - Mean: 0.9266, Std: 0.0039


`observation`
-  Since we used a Random Forest model, scaling techniques did not add much value to the results. 

In [27]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Start MLflow run
with mlflow.start_run(run_name="Baseline model with scaling on selected features"):

    # Log experiment type
    mlflow.log_param("experiment_type", "scaling_on_selected_features")

    # Log model parameters
    mlflow.log_params(rf.get_params())

    # Log evaluation metrics on train and test sets
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log cross-validation results (mean across folds)
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))


🏃 View run Baseline model with scaling on selected features at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/2/runs/c1bbb37fc8ee4b2babec47af46847fa0
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/2


# experiment :03 (Baseline_model + RFECV)

In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 3 - Baseline model and RFECV")

<Experiment: artifact_location='mlflow-artifacts:/0ec061b56cc2469d957f82eca75429ec', creation_time=1754554810638, experiment_id='1', last_update_time=1755070901000, lifecycle_stage='active', name='Exp 3 - Baseline model and RFECV', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X


In [20]:
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode   
'project_name' - constant / top 200 / target encode    
'location' - constant / top 200 / target encode    


'project_in_acres' - KNN Imputer    
'area' - KNN Imputer      
'education_mean_km' - KNN Imputer  
'education_min_km' - KNN Imputer  
'transport_mean_km' - KNN Imputer  
'transport_min_km' - KNN Imputer  
'shopping_centre_mean_km' - KNN Imputer  
'shopping_centre_min_km' - KNN Imputer  
'overall_min_mean_km' - KNN Imputer  
'overall_avg_mean_km' - KNN Imputer  
'overall_min_min_km' - KNN Imputer  
'overall_avg_min_km' - KNN Imputer  
'available_units' - KNN Imputer  
'towers' - KNN Imputer  
'flat_on_floor' - KNN Imputer  
'total_floor' - KNN Imputer  
'bath' - KNN Imputer  
'parking' - KNN Imputer  
'commercial_hub_mean_km' - KNN Imputer  
'commercial_hub_min_km' - KNN Imputer  
'balcony' - KNN Imputer  

'lattitude' - iterative imputer   
'longitude' - iterative imputer   

'lift' - median  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  
'furnish' - mode / ordinal encoding  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ('top_k', TopKCategoriesTransformer(top_k=200)),
    ('target_encoder', TargetEncoder())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 ))
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories))
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation
    (KNNImputer(n_neighbors=5), features_to_fill_knn),
    (IterativeImputer(), features_to_fill_iterative),
    (SimpleImputer(strategy="median"), features_to_fill_median),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

In [25]:
X_train_trans = final_pipeline.fit_transform(X_train,y_train_trans)
X_test_trans = final_pipeline.transform(X_test)

#print(X_train_trans.head())
#print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])


In [26]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

## feature selection using RFECV

In [27]:
# Step Summary:
# 1. Feature selection using RFECV with RandomForestRegressor as the estimator.
# 2. Kept only the selected features from the training and test sets.
# 3. Trained a new RandomForestRegressor on the selected features.
# 4. Evaluated model performance using cross-validation and test metrics.

In [28]:
from sklearn.feature_selection import RFECV

In [29]:
# feature selection using rfecv

rfecv = RFECV(
    estimator=rf,
    step=1,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

In [30]:
# select features

rfecv.fit(X_train_trans, y_train_trans)
print("Selected features:", rfecv.n_features_)
print("Total features:", X_train_trans.shape[1])


Fitting estimator with 83 features.
Fitting estimator with 82 features.
Fitting estimator with 81 features.
Fitting estimator with 80 features.
Fitting estimator with 79 features.
Fitting estimator with 78 features.
Fitting estimator with 77 features.
Fitting estimator with 76 features.
Fitting estimator with 75 features.
Fitting estimator with 74 features.
Fitting estimator with 73 features.
Fitting estimator with 72 features.
Fitting estimator with 71 features.
Fitting estimator with 70 features.
Fitting estimator with 69 features.
Fitting estimator with 68 features.
Fitting estimator with 67 features.
Selected features: 66
Total features: 83


In [31]:
# Check which features were selected
rfecv.get_feature_names_out()

array(['builder', 'project_name', 'location', 'project_in_acres', 'area',
       'education_mean_km', 'education_min_km', 'transport_mean_km',
       'transport_min_km', 'shopping_centre_mean_km',
       'shopping_centre_min_km', 'overall_min_mean_km',
       'overall_avg_mean_km', 'overall_min_min_km', 'overall_avg_min_km',
       'available_units', 'towers', 'flat_on_floor', 'total_floor',
       'bath', 'parking', 'commercial_hub_mean_km',
       'commercial_hub_min_km', 'balcony', 'lattitude', 'longitude',
       'lift', 'property_type_new property', 'property_type_resale',
       'status', 'furnish', 'ownership_co-operative society',
       'ownership_freehold', 'ownership_missing', 'facing_east',
       'facing_missing', 'facing_north - east', 'overlooking_garden/park',
       'overlooking_main road', 'overlooking_missing', 'overlooking_pool',
       'extra_rooms_missing', 'extra_rooms_none of these',
       'extra_rooms_puja', 'extra_rooms_servant', 'extra_rooms_store',
       '

In [32]:
# 1. Transform X data using selected features
X_train_selected = rfecv.transform(X_train_trans)
X_test_selected = rfecv.transform(X_test_trans)



# 2. Train final model
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_selected, y_train_trans)


# 3. Predict (in transformed space)
y_pred_train_trans = final_model.predict(X_train_selected)
y_pred_test_trans = final_model.predict(X_test_selected)
print(y_pred_train_trans.shape)
print(y_pred_test_trans.shape)
print("="*50)

# 4. Clip predictions before inverse transform
min_val, max_val = y_train_trans.min(), y_train_trans.max()
y_pred_train = pt.inverse_transform(np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)).ravel()
y_pred_test = pt.inverse_transform(np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)).ravel()


# 5. Define metric calculation function
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

train_metrics = calc_metrics(y_train, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)


# 6. Cross-validation evaluation
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    final_model,
    X_train_selected,
    y_train_trans,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)


# 7. Print results neatly
print("==== Train Metrics (Selected Features) ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")

print("\n==== Test Metrics (Selected Features) ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")


(9302,)
(2326,)
==== Train Metrics (Selected Features) ====
MAE: 0.4725 | MSE: 3.2309 | RMSE: 1.7975 | R²: 0.8246

==== Test Metrics (Selected Features) ====
MAE: 0.5493 | MSE: 3.0747 | RMSE: 1.7535 | R²: 0.7899

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1396, Std: 0.0006
train_MSE - Mean: 0.0376, Std: 0.0003
train_RMSE - Mean: 0.1939, Std: 0.0008
train_R2 - Mean: 0.9624, Std: 0.0006

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1926, Std: 0.0035
test_MSE - Mean: 0.0713, Std: 0.0029
test_RMSE - Mean: 0.2670, Std: 0.0055
test_R2 - Mean: 0.9285, Std: 0.0045


In [33]:
# # Plot RFECV Scores
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
#          rfecv.cv_results_['mean_test_score'])
# plt.xlabel("Number of Selected Features")
# plt.ylabel("Cross-Validation R² Score")
# plt.title("RFECV Feature Selection")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [34]:
final_model.feature_importances_

array([1.85758138e-02, 2.24971198e-02, 9.49676697e-02, 2.64771536e-03,
       1.73612618e-01, 3.85837693e-03, 3.50955246e-03, 3.30759187e-03,
       2.91666771e-03, 4.12140775e-03, 3.59984704e-03, 4.11299044e-03,
       3.98661242e-03, 4.27994722e-03, 4.00198438e-03, 2.80529202e-03,
       2.52331010e-03, 7.65978442e-03, 2.35793677e-02, 1.22499263e-01,
       3.83655514e-02, 1.14034337e-02, 8.01577448e-03, 2.32446917e-03,
       4.05954862e-02, 3.72113629e-02, 7.22594078e-04, 4.14629018e-04,
       3.75309935e-04, 5.86138854e-04, 1.48638032e-03, 2.43452337e-04,
       4.07664187e-04, 4.92807990e-04, 3.55953311e-04, 5.67695072e-04,
       1.10812884e-04, 3.79690413e-04, 4.59628011e-04, 4.29643457e-04,
       2.49753257e-04, 2.69537138e-03, 2.44635607e-04, 2.59878275e-04,
       8.03905167e-03, 3.72779455e-04, 2.51999653e-04, 1.64378842e-04,
       1.69584280e-03, 7.11116272e-04, 4.72829075e-04, 1.66821886e-03,
       2.65744353e-03, 5.26686455e-04, 8.24113018e-02, 1.77142811e-02,
      

In [35]:
# # feature importance plot

# (
#     pd.DataFrame(final_model.feature_importances_,
#              index=rfecv.transform(X_train_trans).columns,
#              columns=["importance"])
#     .sort_values(by="importance")
#     .plot(kind='barh',figsize=(10,10))
# )

In [36]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Log experiment
with mlflow.start_run(run_name="Baseline model with RFECV"):
    # Log experiment type
    mlflow.log_param("experiment_type", "RFECV")

    # Log model parameters
    mlflow.log_params(final_model.get_params())

    # Log train/test evaluation metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log mean cross-validation metrics
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))

🏃 View run Baseline model with RFECV at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/1/runs/88ae6a41b1ed42098e9dd9875f3445c3
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/1


# experiment :04 (Baseline_model + simple LOF + RFECV)

In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 4 - Baseline model with LOF and RFECV")

<Experiment: artifact_location='mlflow-artifacts:/001eaafdb27d4025892882de70dbc664', creation_time=1755072897895, experiment_id='6', last_update_time=1755072897895, lifecycle_stage='active', name='Exp 4 - Baseline model with LOF and RFECV', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X


In [20]:
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode   
'project_name' - constant / top 200 / target encode    
'location' - constant / top 200 / target encode    


'project_in_acres' - KNN Imputer    
'area' - KNN Imputer      
'education_mean_km' - KNN Imputer  
'education_min_km' - KNN Imputer  
'transport_mean_km' - KNN Imputer  
'transport_min_km' - KNN Imputer  
'shopping_centre_mean_km' - KNN Imputer  
'shopping_centre_min_km' - KNN Imputer  
'overall_min_mean_km' - KNN Imputer  
'overall_avg_mean_km' - KNN Imputer  
'overall_min_min_km' - KNN Imputer  
'overall_avg_min_km' - KNN Imputer  
'available_units' - KNN Imputer  
'towers' - KNN Imputer  
'flat_on_floor' - KNN Imputer  
'total_floor' - KNN Imputer  
'bath' - KNN Imputer  
'parking' - KNN Imputer  
'commercial_hub_mean_km' - KNN Imputer  
'commercial_hub_min_km' - KNN Imputer  
'balcony' - KNN Imputer  

'lattitude' - iterative imputer   
'longitude' - iterative imputer   

'lift' - median  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  
'furnish' - mode / ordinal encoding  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
print(X_train.shape)

(9302, 47)


In [26]:
# Here, only delete the rows that are outliers in the below columns.
# Even if imputation is applied below, it does NOT impute permanently —
# it only imputes temporarily to help detect and delete outlier data points.

features_to_fill_knn_OD  = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',                           
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
]  #OD  means outlier detetion

features_to_fill_iterative_OD = ['lattitude','longitude']

# Impute OD columns separately for LOF
knn_imputer = KNNImputer(n_neighbors=5)
X_train_knn_imp = X_train[features_to_fill_knn_OD].copy()
X_train_knn_imp.loc[:, :] = knn_imputer.fit_transform(X_train_knn_imp)

iter_imputer = IterativeImputer()
X_train_iter_imp = X_train[features_to_fill_iterative_OD].copy()
X_train_iter_imp.loc[:, :] = iter_imputer.fit_transform(X_train_iter_imp)

# Apply LOF separately
lof_knn = LocalOutlierFactor(n_neighbors=20, contamination=0.01)
lof_iter = LocalOutlierFactor(n_neighbors=20, contamination=0.01)

mask_knn = (lof_knn.fit_predict(X_train_knn_imp) == 1)
mask_iter = (lof_iter.fit_predict(X_train_iter_imp) == 1)

final_mask = mask_knn & mask_iter

# Filter train data and target
X_train_filtered = X_train.loc[final_mask].copy()
y_train_filtered = y_train_trans.loc[final_mask].copy()

print(f"Original train size: {X_train.shape[0]}")
print(f"Filtered train size after LOF: {X_train_filtered.shape[0]}")

Original train size: 9302
Filtered train size after LOF: 9114


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neighbors\_lof.py:322: UserWarning: Duplicate values are leading to incorrect results. Increase the number of neighbors for more accurate results.
  warnings.warn(


In [25]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ('top_k', TopKCategoriesTransformer(top_k=200)),
    ('target_encoder', TargetEncoder())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 ))
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories))
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation
    (KNNImputer(n_neighbors=5), features_to_fill_knn),
    (IterativeImputer(), features_to_fill_iterative),
    (SimpleImputer(strategy="median"), features_to_fill_median),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

In [27]:
X_train_trans = final_pipeline.fit_transform(X_train_filtered, y_train_filtered)
X_test_trans  = final_pipeline.transform(X_test)

#print(X_train_trans.head())
print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])

Series([], dtype: int64)


In [28]:
print(X_train_trans.shape)
print(y_train_filtered.shape)

(9114, 83)
(9114,)


In [29]:
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

## feature selection using RFECV

In [30]:
# Step Summary:
# 1. Feature selection using RFECV with RandomForestRegressor as the estimator.
# 2. Kept only the selected features from the training and test sets.
# 3. Trained a new RandomForestRegressor on the selected features.
# 4. Evaluated model performance using cross-validation and test metrics.

In [31]:
from sklearn.feature_selection import RFECV

In [32]:
# feature selection using rfecv

rfecv = RFECV(
    estimator=rf,
    step=1,
    cv=5,
    scoring='r2',
    n_jobs=-1,
    verbose=2
)

In [33]:
# select features

rfecv.fit(X_train_trans, y_train_filtered)

Fitting estimator with 83 features.
Fitting estimator with 82 features.
Fitting estimator with 81 features.
Fitting estimator with 80 features.
Fitting estimator with 79 features.
Fitting estimator with 78 features.
Fitting estimator with 77 features.
Fitting estimator with 76 features.
Fitting estimator with 75 features.
Fitting estimator with 74 features.
Fitting estimator with 73 features.
Fitting estimator with 72 features.
Fitting estimator with 71 features.
Fitting estimator with 70 features.
Fitting estimator with 69 features.
Fitting estimator with 68 features.
Fitting estimator with 67 features.
Fitting estimator with 66 features.
Fitting estimator with 65 features.
Fitting estimator with 64 features.
Fitting estimator with 63 features.
Fitting estimator with 62 features.
Fitting estimator with 61 features.


RFECV(cv=5, estimator=RandomForestRegressor(n_jobs=-1, random_state=42),
      n_jobs=-1, scoring='r2', verbose=2)

In [34]:
# Check which features were selected
rfecv.get_feature_names_out()

array(['builder', 'project_name', 'location', 'project_in_acres', 'area',
       'education_mean_km', 'education_min_km', 'transport_mean_km',
       'transport_min_km', 'shopping_centre_mean_km',
       'shopping_centre_min_km', 'overall_min_mean_km',
       'overall_avg_mean_km', 'overall_min_min_km', 'overall_avg_min_km',
       'available_units', 'towers', 'flat_on_floor', 'total_floor',
       'bath', 'parking', 'commercial_hub_mean_km',
       'commercial_hub_min_km', 'balcony', 'lattitude', 'longitude',
       'lift', 'property_type_new property', 'status', 'furnish',
       'ownership_co-operative society', 'ownership_freehold',
       'ownership_missing', 'facing_east', 'facing_missing',
       'facing_north - east', 'overlooking_garden/park',
       'overlooking_main road', 'overlooking_missing',
       'extra_rooms_missing', 'extra_rooms_puja', 'extra_rooms_servant',
       'extra_rooms_store', 'extra_rooms_study', 'flooring_ceramic tiles',
       'flooring_marble', 'floorin

In [35]:
# 1. Transform X data using selected features
X_train_selected = rfecv.transform(X_train_trans)
X_test_selected = rfecv.transform(X_test_trans)


# 2. Train final model
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_selected, y_train_filtered)


# 3. Predict (in transformed space)
y_pred_train_trans = final_model.predict(X_train_selected)
y_pred_test_trans = final_model.predict(X_test_selected)


# 4. Clip predictions before inverse transform
min_val, max_val = y_train_filtered.min(), y_train_filtered.max()
y_pred_train = pt.inverse_transform(np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)).ravel()
y_pred_test = pt.inverse_transform(np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)).ravel()

y_train_filtered_inv = pt.inverse_transform(y_train_filtered.values.reshape(-1, 1)).ravel()


# 5. Define metric calculation function
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2


train_metrics = calc_metrics(y_train_filtered_inv, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)


# 6. Cross-validation evaluation
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    final_model,
    X_train_selected,
    y_train_filtered,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)


# 7. Print results neatly
print("==== Train Metrics (Selected Features) ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")

print("\n==== Test Metrics (Selected Features) ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

==== Train Metrics (Selected Features) ====
MAE: 0.4594 | MSE: 2.5281 | RMSE: 1.5900 | R²: 0.8476

==== Test Metrics (Selected Features) ====
MAE: 0.5569 | MSE: 3.1765 | RMSE: 1.7823 | R²: 0.7829

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1414, Std: 0.0010
train_MSE - Mean: 0.0383, Std: 0.0005
train_RMSE - Mean: 0.1957, Std: 0.0012
train_R2 - Mean: 0.9612, Std: 0.0008

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1942, Std: 0.0029
test_MSE - Mean: 0.0723, Std: 0.0023
test_RMSE - Mean: 0.2689, Std: 0.0043
test_R2 - Mean: 0.9265, Std: 0.0042


In [36]:
# # Plot RFECV Scores
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
#          rfecv.cv_results_['mean_test_score'])
# plt.xlabel("Number of Selected Features")
# plt.ylabel("Cross-Validation R² Score")
# plt.title("RFECV Feature Selection")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [37]:
final_model.feature_importances_

array([2.32499632e-02, 2.26887954e-02, 8.87162237e-02, 2.90168912e-03,
       1.61300223e-01, 4.61266007e-03, 3.62212182e-03, 3.43791313e-03,
       3.06738385e-03, 4.18391959e-03, 3.00591364e-03, 4.20444562e-03,
       4.01329178e-03, 3.76696304e-03, 4.75513403e-03, 2.83404504e-03,
       2.99275693e-03, 8.71445535e-03, 2.39751240e-02, 1.29643647e-01,
       3.39608718e-02, 1.07196657e-02, 8.48016924e-03, 2.51442968e-03,
       3.26419318e-02, 4.13267385e-02, 1.47581955e-03, 4.79316187e-04,
       4.89428891e-04, 1.47629116e-03, 2.70179827e-04, 3.82045696e-04,
       7.09362406e-04, 3.46049795e-04, 8.23366218e-04, 1.24083873e-04,
       4.11833720e-04, 4.55958674e-04, 5.60668435e-04, 3.19328251e-03,
       3.72980409e-04, 7.82725260e-03, 6.40870143e-04, 4.47327273e-04,
       1.36711331e-04, 1.36275506e-03, 1.18642399e-03, 4.65396474e-04,
       2.19301816e-03, 2.76032268e-03, 7.45624847e-04, 8.66160454e-02,
       2.00304242e-02, 6.95289592e-03, 1.27138303e-02, 9.87106819e-03,
      

In [38]:
# # feature importance plot

# (
#     pd.DataFrame(final_model.feature_importances_,
#              index=rfecv.transform(X_train_trans).columns,
#              columns=["importance"])
#     .sort_values(by="importance")
#     .plot(kind='barh',figsize=(10,10))
# )

In [39]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Log experiment
with mlflow.start_run(run_name="Baseline model with LOF and RFECV"):
    # Log experiment type
    mlflow.log_param("experiment_type", "LOF")

    # Log model parameters
    mlflow.log_params(final_model.get_params())

    # Log train/test evaluation metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log mean cross-validation metrics
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))

🏃 View run Baseline model with LOF and RFECV at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/6/runs/c40de961feba49318c02f6a5a6889cf2
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/6


# experiment :05 (Baseline_model + permutation importance)

In [15]:
# mlflow experiment

mlflow.set_experiment("Exp 5 - Baseline model and permutation importance")

<Experiment: artifact_location='mlflow-artifacts:/b9fca074291a4addb924ae795ffa77b8', creation_time=1755143214950, experiment_id='7', last_update_time=1755143214950, lifecycle_stage='active', name='Exp 5 - Baseline model and permutation importance', tags={}>

In [16]:
# --- Custom MultiLabel Binarizer ---
class MultiLabelBinarizerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoders = {}
        self.columns = []

    def fit(self, X, y=None):
        self.columns = X.columns
        for col in self.columns:
            mlb = MultiLabelBinarizer()
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            mlb.fit(split_data)
            self.encoders[col] = mlb
        return self

    def transform(self, X):
        output_parts = []
        for col in self.columns:
            mlb = self.encoders[col]
            split_data = X[col].apply(lambda x: x.split(', ') if isinstance(x, str) else [])
            transformed = mlb.transform(split_data)
            col_names = [f"{col}_{cls}" for cls in mlb.classes_]
            output_parts.append(pd.DataFrame(transformed, columns=col_names, index=X.index))
        return pd.concat(output_parts, axis=1)

In [17]:
# --- Custom Transformer: Top K Categories ---
class TopKCategoriesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, top_k=200):
        self.top_k = top_k
        self.top_categories_ = {}

    def fit(self, X, y=None):
        for col in X.columns:
            top = X[col].value_counts().nlargest(self.top_k).index
            self.top_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in X.columns:
            X[col] = X[col].where(X[col].isin(self.top_categories_[col]), other='__other__')
        return X

In [18]:
#final
amenities_weightages = {
    "sea facing": 10,
    "private pool": 10,
    "private jaccuzi": 10,
    "sky villa": 10,
    "helipad": 10,
    "wrap around balcony": 7,
    "infinity swimming pool": 10,
    "high ceiling": 9,
    "located in the heart of city": 10,
    "large open space": 10,
    "skyline view": 10,
    "private terrace/garden": 10,
    "private garage": 10,
    "mansion": 10,
    "club house": 9,
    "large clubhouse": 9,
    "modular kitchen": 9,
    "central ac": 9,
    "banquet hall": 6,
    "premium branded fittings": 9,
    "private garden": 9,
    "full glass wall": 9,
    "garden view": 9,
    "theme based architectures": 9,
    "grand entrance lobby": 9,
    "smart home": 9,
    "library and business centre": 9,
    "recreational pool": 9,
    "projector": 8,
    "swimming pool": 8,
    "gymnasium": 8,
    "indoor squash & badminton courts": 8,
    "outdoor tennis courts": 8,
    "cycling & jogging track": 8,
    "kids play pool with water slides": 8,
    "guest lobby in each floor": 8,
    "aesthetically designed landscape garden": 8,
    "health club with steam / jacuzzi": 8,
    "meditation area": 8,
    "pet park": 8,
    "visitor parking": 8,
    "badminton court": 8,
    "kids play area": 7,
    "community hall": 7,
    "power back up": 7,
    "cctv camera": 7,
    "rain water harvesting": 7,
    "internet/wi-fi connectivity": 7,
    "cycling track": 7,
    "art center": 7,
    "library": 7,
    "fire sprinklers": 7,
    "multipurpose hall": 7,
    "event space & amphitheatre": 7,
    "flower gardens": 6,
    "curated garden": 6,
    "multipurpose courts": 7,
    "dth television facility": 5,
    "fire fighting equipment": 6,
    "provision for power backup": 7,
    "sand pit": 6,
    "sewage treatment plant": 6,
    "solar energy": 7,
    "piped gas": 6,
    "kids club": 6,
    "waste disposal": 6,
    "lift": 5,
    "security": 5,
    "maintenance staff": 5,
    "reserved parking": 5,
    "ro water system": 5,
    "wheelchair accessibility": 5,
    "shopping center": 5,
    "laundry service": 5,
    "bank & atm": 5,
    "community entrance gate": 5,
    "canopy walk": 4,
    "entry exit gate": 4,
    "early learning centre": 4,
    "earth quake resistant": 7,
    "waste water recycling": 6,
    "whiteboard": 3,
    "printer": 3,
    "tea/coffee": 3,
    "house help accommodation": 7,
    "study room": 5,
    "ground water recharging": 5,
    "unknown": 0,
    "3 tier security system": 8,
    "ac in each room": 9,
    "activity deck4": 7,
    "aerobics room": 7,
    "air conditioned": 9,
    "all wooden flooring": 8,
    "arts & craft studio": 6,
    "bar/lounge": 7,
    "barbeque pit": 6,
    "barbeque space": 6,
    "cafeteria/food court": 7,
    "coffee lounge & restaurants": 7,
    "concierge services": 9,
    "conference room": 8,
    "cricket net practice": 6,
    "dance studio": 7,
    "downtown": 10,
    "fingerprint access": 8,
    "fireplace": 6,
    "golf course": 10,
    "hilltop": 10,
    "horticulture": 6,
    "indoor games room": 7,
    "island kitchen layout": 8,
    "jogging and strolling track": 7,
    "kids splash pool": 7,
    "lawn with pathway": 6,
    "guest accommodation":8,
    "marble flooring": 9,
    "mini cinema theatre": 9,
    "half basketball court":7,
    "park": 8,
    "pool with temperature control": 10,
    "intercom facility":6,
    "rentable community space": 6,
    "retail boulevard (retail shops)": 8,
    "service/goods lift": 6,
    "skydeck": 9,
    "vaastu compliant": 7,
    "volleyball court": 6,
    "water front": 10,
    "water storage": 5,
    "water treatment plant": 7,
    "wine cellar": 8
}

In [19]:
class AmenitiesScoreTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, column='amenities', weightages=None, output_column='assigned_amenities_score'):
        self.column = column
        self.weightages = weightages if weightages is not None else {}
        self.output_column = output_column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        def calculate_score(amenities_str):
            # if not isinstance(amenities_str, str):
            #     return 0
            # amenities_list = [a.strip().lower() for a in amenities_str.split(",")]
            # return round(sum(self.weightages.get(a, 0) for a in amenities_list), 2)
            amenities_types = [f.strip().lower() for f in amenities_str.split(",")]
            total_weight = sum(self.weightages.get(f, 0) for f in amenities_types)
            return round(total_weight, 2)

        # Calculate scores
        X[self.output_column] = X[self.column].apply(calculate_score)

        # Replace 0 with NaN
        X[self.output_column] = X[self.output_column].replace(0, pd.NA)
        X[self.output_column] = pd.to_numeric(X[self.output_column], errors='coerce')

        # Drop original amenities column
        X.drop(columns=[self.column], inplace=True)

        return X


In [20]:
# Add missing indicator
class MissingIndicatorAdder(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column + '_missing'] = X[self.column].isna().astype(int)
        return X

In [21]:
# KNN imputation + MinMax scaling
class ImputeAndScaleAmenity(BaseEstimator, TransformerMixin):
    def __init__(self, column='assigned_amenities_score', n_neighbors=5):
        self.column = column
        self.n_neighbors = n_neighbors
        self.imputer = KNNImputer(n_neighbors=n_neighbors)
        self.scaler = StandardScaler()

    def fit(self, X, y=None):
        # Fit on the original column
        self.imputer.fit(X[[self.column]])
        imputed = self.imputer.transform(X[[self.column]])
        self.scaler.fit(imputed)
        return self

    def transform(self, X):
        X = X.copy()
        # Impute and scale the column
        imputed = self.imputer.transform(X[[self.column]])
        scaled = self.scaler.transform(imputed)
        # Replace with scaled
        X[self.column] = scaled
        return X

In [22]:
# Custom Ordinal Encoder Wrapper (for single column)
class ConstructionOrdinalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, column='construction', categories=None):
        self.column = column
        self.categories = categories
        self.encoder = OrdinalEncoder(categories=self.categories, handle_unknown='use_encoded_value', unknown_value=-1)

    def fit(self, X, y=None):
        self.encoder.fit(X[[self.column]])
        return self

    def transform(self, X):
        X = X.copy()
        X[self.column] = self.encoder.transform(X[[self.column]])
        return X

'builder' - constant / top 200 / target encode   
'project_name' - constant / top 200 / target encode    
'location' - constant / top 200 / target encode    


'project_in_acres' - KNN Imputer    
'area' - KNN Imputer      
'education_mean_km' - KNN Imputer  
'education_min_km' - KNN Imputer  
'transport_mean_km' - KNN Imputer  
'transport_min_km' - KNN Imputer  
'shopping_centre_mean_km' - KNN Imputer  
'shopping_centre_min_km' - KNN Imputer  
'overall_min_mean_km' - KNN Imputer  
'overall_avg_mean_km' - KNN Imputer  
'overall_min_min_km' - KNN Imputer  
'overall_avg_min_km' - KNN Imputer  
'available_units' - KNN Imputer  
'towers' - KNN Imputer  
'flat_on_floor' - KNN Imputer  
'total_floor' - KNN Imputer  
'bath' - KNN Imputer  
'parking' - KNN Imputer  
'commercial_hub_mean_km' - KNN Imputer  
'commercial_hub_min_km' - KNN Imputer  
'balcony' - KNN Imputer  

'lattitude' - iterative imputer   
'longitude' - iterative imputer   

'lift' - median  

'property_type' - mode / OHE  
'status' - mode / ordinal encoding  
'furnish' - mode / ordinal encoding  


'ownership' - constant / OHE  
'facing' - constant / OHE  
'overlooking' - constant / multilable  
'extra_rooms' - constant / multilable  
'flooring' - constant / multilable  


'assigned_amenities_score' -  missingindicator then KNNimputation and then min_max_scale
'construction' -  missingindicator and ordinal_encode

'city' - OHE  
'seller' - OHE  


'education_within_2km' - MinMax Scaling    
'transport_within_2km' - MinMax Scaling     
'shopping_centre_within_2km' - MinMax Scaling    
'commercial_hub_within_2km' - MinMax Scaling  
'hospital_within_2km' - MinMax Scaling  
'tourist_within_2km' - MinMax Scaling  
'total_within_2km' - MinMax Scaling  

In [23]:
impute_topk_target_encoding_cols = ['builder', 'project_name', 'location']

features_to_fill_knn = [
    'project_in_acres','area', 'education_mean_km', 'education_min_km',
    'transport_mean_km','transport_min_km','shopping_centre_mean_km','shopping_centre_min_km',
    'overall_min_mean_km', 'overall_avg_mean_km','overall_min_min_km', 'overall_avg_min_km',
    'available_units','towers','flat_on_floor','total_floor','bath','parking',
    'commercial_hub_mean_km','commercial_hub_min_km','balcony'
] #'costpersqft','emi'

features_to_fill_iterative = ['lattitude','longitude']
features_to_fill_median = ['lift']

impute_mf_and_OHE = ['property_type']

impute_mf_and_ordinal_encode = ['status','furnish']
ordinal_categories = [
    ['under construction', 'ongoing', 'ready to move'],  # status
    ['unfurnished', 'semi-furnished', 'furnished']       # furnish
]

impute_missing_and_OHE = ['ownership', 'facing']

impute_missing_and_multilable = ['overlooking','extra_rooms','flooring']

assignweight_missingindicator_KNNimputation_minmaxscale = ['amenities']

missingindicator_ordinal_encode = ['construction']
construction_categories = [[
    'missing', 'under construction', 'new construction', 'less than 5 years',
    '5 to 10 years', '10 to 15 years', '15 to 20 years', 'above 20 years'
]]

onehotencode = ['city','seller']

min_max_scaling = ['education_within_2km','transport_within_2km','shopping_centre_within_2km',
                   'commercial_hub_within_2km','hospital_within_2km','tourist_within_2km','total_within_2km']

In [24]:
# --- Final Pipeline for Target Encoding Columns ---
builder_location_project_name_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ('top_k', TopKCategoriesTransformer(top_k=200)),
    ('target_encoder', TargetEncoder())
])

property_type_pipeline = Pipeline(steps=[
    ('impute' , SimpleImputer(strategy="most_frequent")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


status_furnish_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="most_frequent")),
    ('ordinal encode', OrdinalEncoder(categories=ordinal_categories,handle_unknown='use_encoded_value',unknown_value=-1 ))
])


ownership_facing_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('OHE', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

overlooking_extra_rooms_flooring_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy="constant", fill_value="missing")),
    ('multilable', MultiLabelBinarizerTransformer())
])

assigned_amenities_pipeline = Pipeline(steps=[
    ('calculate_score', AmenitiesScoreTransformer(
        column='amenities',
        weightages=amenities_weightages,
        output_column='assigned_amenities_score'
    )),
    ('add_missing_indicator', MissingIndicatorAdder(column='assigned_amenities_score')),
    ('impute_and_scale', ImputeAndScaleAmenity(column='assigned_amenities_score', n_neighbors=5))
])

construction_pipeline = Pipeline(steps=[
    ('add_missing_indicator', MissingIndicatorAdder(column='construction')),
    ('ordinal_encode', ConstructionOrdinalEncoder(column='construction', categories=construction_categories))
])

city_seller_pipeline = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

within2km_pipeline = Pipeline(steps=[
    ("scaler", StandardScaler())
])

# --- Unified Preprocessor ---
preprocessor = make_column_transformer(
    # impute_constant=missing, topK and Target Encoding
    (builder_location_project_name_pipeline, impute_topk_target_encoding_cols),

    # Imputation
    (KNNImputer(n_neighbors=5), features_to_fill_knn),
    (IterativeImputer(), features_to_fill_iterative),
    (SimpleImputer(strategy="median"), features_to_fill_median),

    # impute = most_frequent and OHE 
    (property_type_pipeline, impute_mf_and_OHE),

    #impute = most_frequent and ordinal encoding
    (status_furnish_pipeline, impute_mf_and_ordinal_encode),

    #impute_constant=missing and OHE
    (ownership_facing_pipeline, impute_missing_and_OHE),

    #impute_constant=missing and multilable
    (overlooking_extra_rooms_flooring_pipeline, impute_missing_and_multilable),

    #missingindicator then KNNimputation and then min_max_scale
    (assigned_amenities_pipeline, assignweight_missingindicator_KNNimputation_minmaxscale),

    #missingindicator and ordinal_encode
    (construction_pipeline, missingindicator_ordinal_encode),

    # onehot_encoder
    (city_seller_pipeline, onehotencode),

    # MinMax Scaling
    (within2km_pipeline, min_max_scaling),

    # Keep other columns
    remainder='passthrough',
    verbose_feature_names_out=False,
    n_jobs=-1
)

# --- Final Pipeline ---
final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor)
])

In [25]:
X_train_trans = final_pipeline.fit_transform(X_train,y_train_trans)
X_test_trans = final_pipeline.transform(X_test)

#print(X_train_trans.head())
#print(X_train_trans.isna().sum()[X_train_trans.isna().sum() > 0])


## feature selection using permutation importance

In [26]:
# 2. Train final model
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_train_trans, y_train_trans)

from sklearn.inspection import permutation_importance
import numpy as np
import pandas as pd

# --- Get feature names from pipeline ---
try:
    # If your pipeline ends with a ColumnTransformer
    feature_names = final_pipeline.get_feature_names_out()
except AttributeError:
    # Fallback if using manual names
    feature_names = [f"f{i}" for i in range(X_train_trans.shape[1])]

# --- Calculate permutation importance ---
perm_result = permutation_importance(
    final_model,
    X_train_trans,
    y_train_trans,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)


# --- Sort by importance ---
sorted_idx = np.argsort(perm_result.importances_mean)[::-1]

print("=== Permutation Importance Rankings ===")
for idx in sorted_idx:
    print(f"{feature_names[idx]}: Mean Importance = {perm_result.importances_mean[idx]:.6f}, "
          f"Std = {perm_result.importances_std[idx]:.6f}")

# --- Identify low-importance features ---
threshold = 0.0001  # Adjust as needed
low_importance_indices = [idx for idx in sorted_idx if perm_result.importances_mean[idx] <= threshold]
low_importance_features = [feature_names[i] for i in low_importance_indices]

print("\n=== Features with Near-Zero Importance ===")
print(low_importance_features)

# Create new DataFrame with only low-importance features
low_importance_df = X_train_trans.iloc[:, low_importance_indices]
low_importance_df.columns = low_importance_features

print(f"\nShape of low-importance feature DataFrame: {low_importance_df.shape}")



=== Permutation Importance Rankings ===
f4: Mean Importance = 0.112476, Std = 0.001416
f82: Mean Importance = 0.097367, Std = 0.001493
f68: Mean Importance = 0.061674, Std = 0.001273
f19: Mean Importance = 0.057124, Std = 0.001177
f2: Mean Importance = 0.041145, Std = 0.000812
f25: Mean Importance = 0.025107, Std = 0.000595
f24: Mean Importance = 0.021912, Std = 0.000147
f18: Mean Importance = 0.009689, Std = 0.000151
f1: Mean Importance = 0.009267, Std = 0.000328
f70: Mean Importance = 0.008791, Std = 0.000219
f0: Mean Importance = 0.008439, Std = 0.000234
f20: Mean Importance = 0.008205, Std = 0.000158
f73: Mean Importance = 0.004582, Std = 0.000078
f17: Mean Importance = 0.004146, Std = 0.000101
f71: Mean Importance = 0.004127, Std = 0.000163
f74: Mean Importance = 0.004079, Std = 0.000088
f21: Mean Importance = 0.004006, Std = 0.000070
f81: Mean Importance = 0.003995, Std = 0.000099
f22: Mean Importance = 0.003982, Std = 0.000083
f69: Mean Importance = 0.003160, Std = 0.000116
f11:

In [27]:
# === 1. Drop Low-Importance Features ===
# Drop by indices instead of names
X_train_final = X_train_trans.drop(X_train_trans.columns[low_importance_indices], axis=1)
X_test_final = X_test_trans.drop(X_test_trans.columns[low_importance_indices], axis=1)

print(f"Reduced features: {X_train_final.shape[1]} from {X_train_trans.shape[1]}")


# === 2. Retrain Final Model ===
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train_final, y_train_trans)

# === 3. Predict (in transformed space) ===
y_pred_train_trans = final_model.predict(X_train_final)
y_pred_test_trans = final_model.predict(X_test_final)

# === 4. Clip predictions before inverse transform ===
min_val, max_val = y_train_trans.min(), y_train_trans.max()
y_pred_train = pt.inverse_transform(
    np.clip(y_pred_train_trans, min_val, max_val).reshape(-1, 1)
).ravel()
y_pred_test = pt.inverse_transform(
    np.clip(y_pred_test_trans, min_val, max_val).reshape(-1, 1)
).ravel()

# === 5. Define Metric Calculation Function ===
def calc_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    return mae, mse, rmse, r2

train_metrics = calc_metrics(y_train, y_pred_train)
test_metrics = calc_metrics(y_test, y_pred_test)

# === 6. Cross-validation Evaluation ===
scoring = {
    'MAE': make_scorer(mean_absolute_error),
    'MSE': make_scorer(mean_squared_error),
    'RMSE': make_scorer(lambda y, y_pred: np.sqrt(mean_squared_error(y, y_pred))),
    'R2': make_scorer(r2_score)
}

cv_results = cross_validate(
    final_model,
    X_train_final,
    y_train_trans,
    scoring=scoring,
    cv=5,
    return_train_score=True,
    n_jobs=-1
)

# === 7. Print Results Neatly ===
print("==== Train Metrics (Reduced Features) ====")
print(f"MAE: {train_metrics[0]:.4f} | MSE: {train_metrics[1]:.4f} | RMSE: {train_metrics[2]:.4f} | R²: {train_metrics[3]:.4f}")

print("\n==== Test Metrics (Reduced Features) ====")
print(f"MAE: {test_metrics[0]:.4f} | MSE: {test_metrics[1]:.4f} | RMSE: {test_metrics[2]:.4f} | R²: {test_metrics[3]:.4f}")

print("\n==== Cross-Validation Train Scores ====")
for key, value in cv_results.items():
    if key.startswith("train"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")

print("\n==== Cross-Validation Test Scores ====")
for key, value in cv_results.items():
    if key.startswith("test"):
        print(f"{key} - Mean: {np.mean(value):.4f}, Std: {np.std(value):.4f}")


Reduced features: 68 from 83
==== Train Metrics (Reduced Features) ====
MAE: 0.4697 | MSE: 3.1007 | RMSE: 1.7609 | R²: 0.8317

==== Test Metrics (Reduced Features) ====
MAE: 0.5467 | MSE: 3.0564 | RMSE: 1.7483 | R²: 0.7911

==== Cross-Validation Train Scores ====
train_MAE - Mean: 0.1406, Std: 0.0008
train_MSE - Mean: 0.0381, Std: 0.0005
train_RMSE - Mean: 0.1951, Std: 0.0012
train_R2 - Mean: 0.9619, Std: 0.0007

==== Cross-Validation Test Scores ====
test_MAE - Mean: 0.1931, Std: 0.0036
test_MSE - Mean: 0.0716, Std: 0.0028
test_RMSE - Mean: 0.2675, Std: 0.0052
test_R2 - Mean: 0.9283, Std: 0.0044


In [28]:
# # Plot RFECV Scores
# plt.figure(figsize=(10, 5))
# plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
#          rfecv.cv_results_['mean_test_score'])
# plt.xlabel("Number of Selected Features")
# plt.ylabel("Cross-Validation R² Score")
# plt.title("RFECV Feature Selection")
# plt.grid(True)
# plt.tight_layout()
# plt.show()


In [29]:
final_model.feature_importances_

array([0.02931782, 0.02456591, 0.07190776, 0.00272785, 0.17054149,
       0.0041042 , 0.00331949, 0.00332859, 0.00304843, 0.00452421,
       0.0031583 , 0.00390616, 0.00334689, 0.00340044, 0.00452666,
       0.00271284, 0.00250512, 0.00791218, 0.02316557, 0.12064139,
       0.03719003, 0.00984921, 0.00873906, 0.00236295, 0.04486515,
       0.0403733 , 0.001387  , 0.00040208, 0.00041335, 0.000471  ,
       0.00126408, 0.00024305, 0.00040709, 0.00046803, 0.00036597,
       0.00058364, 0.00034925, 0.00042089, 0.00054032, 0.00027657,
       0.00269845, 0.00020577, 0.00047796, 0.00822811, 0.00033434,
       0.00042846, 0.00017443, 0.00197484, 0.00081612, 0.00054572,
       0.00180505, 0.0004977 , 0.0025465 , 0.00061547, 0.09378683,
       0.00661449, 0.01873467, 0.00926311, 0.0072979 , 0.00784086,
       0.00237152, 0.0007841 , 0.00241755, 0.00154397, 0.00081226,
       0.00052365, 0.01004251, 0.17298434])

In [30]:
# # feature importance plot

# (
#     pd.DataFrame(final_model.feature_importances_,
#              index=rfecv.transform(X_train_trans).columns,
#              columns=["importance"])
#     .sort_values(by="importance")
#     .plot(kind='barh',figsize=(10,10))
# )

In [31]:
# Unpack metrics
train_mae, train_mse, train_rmse, train_r2 = train_metrics
test_mae, test_mse, test_rmse, test_r2 = test_metrics

# Log experiment
with mlflow.start_run(run_name="Baseline model with permutation importance"):
    # Log experiment type
    mlflow.log_param("experiment_type", "permutation importance")

    # Log model parameters
    mlflow.log_params(final_model.get_params())

    # Log train/test evaluation metrics
    mlflow.log_metric("train_mae", train_mae)
    mlflow.log_metric("train_mse", train_mse)
    mlflow.log_metric("train_rmse", train_rmse)
    mlflow.log_metric("train_r2", train_r2)

    mlflow.log_metric("test_mae", test_mae)
    mlflow.log_metric("test_mse", test_mse)
    mlflow.log_metric("test_rmse", test_rmse)
    mlflow.log_metric("test_r2", test_r2)

    # Log mean cross-validation metrics
    mlflow.log_metric("cv_train_mae", np.mean(cv_results['train_MAE']))
    mlflow.log_metric("cv_train_mse", np.mean(cv_results['train_MSE']))
    mlflow.log_metric("cv_train_rmse", np.mean(cv_results['train_RMSE']))
    mlflow.log_metric("cv_train_r2", np.mean(cv_results['train_R2']))

    mlflow.log_metric("cv_val_mae", np.mean(cv_results['test_MAE']))
    mlflow.log_metric("cv_val_mse", np.mean(cv_results['test_MSE']))
    mlflow.log_metric("cv_val_rmse", np.mean(cv_results['test_RMSE']))
    mlflow.log_metric("cv_val_r2", np.mean(cv_results['test_R2']))

🏃 View run Baseline model with permutation importance at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/7/runs/22a5b7839ae84567b6a888e16c427b56
🧪 View experiment at: https://dagshub.com/bowlekarbhushan88/property-price-prediction.mlflow/#/experiments/7


`observation`
- so experiment :03 (Baseline_model + RFECV) and experiment :05 (Baseline_model + permutation importance) are almost tied